# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the dataset [Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells](https://sen.science/doi/10.71728/senscience.vq4a-28xa) using the [`mlcroissant`](https://github.com/mlcommons/croissant) library, referencing entities by their `@id` fields for clarity and reproducibility.

### Dataset Source
The dataset is described by a Croissant schema JSON-LD file, available at:

```
https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json
```


In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load the metadata and dataset record sets using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"
dataset = mlc.Dataset(croissant_url)

# Print key facets from the metadata
print(f"Dataset Name: {dataset.metadata.name}")
print(f"Version: {dataset.metadata.version}")
print(f"Description: {dataset.metadata.description}")
print(f"License: {dataset.metadata.license}")
print(f"Citation: {dataset.metadata.citeAs}")

## 2. Data Overview
Review available record sets, their fields, and columns referenced by their `@id`s.

> **Note:** All references to record sets, fields, etc., are by `@id` as required by the Croissant specification.

In [ ]:
# List all RecordSets by @id and show field IDs for each
record_sets = dataset.metadata.record_set
print("Available Record Sets and their fields/columns (@id):\n")
record_set_ids = []
for rs in record_sets:
    print(f"- RecordSet @id: {rs.id}")
    record_set_ids.append(rs.id)
    if hasattr(rs, 'field') and rs.field:
        fields = rs.field if isinstance(rs.field, list) else [rs.field]
        print("  Fields:")
        for fld in fields:
            print(f"    - Field @id: {fld.id} (name: {fld.name})")
    if hasattr(rs, 'column') and rs.column:
        columns = rs.column if isinstance(rs.column, list) else [rs.column]
        print("  Columns:")
        for col in columns:
            print(f"    - Column @id: {col.id} (name: {col.name})")
    print()

## 3. Data Extraction
Extract the data for each record set using their `@id` and load it into a pandas DataFrame for analysis. Only a preview of each is shown.

We will systematically load each available record set, keying by its `@id` per best Croissant practice.

In [ ]:
dataframes = {}
# Choose a RecordSet @id (for demonstration, we take the first one)
if len(record_set_ids) == 0:
    print("No record sets found in the dataset.")
else:
    for record_set_id in record_set_ids:
        # Each yielded record is a dict where keys are field/column @id
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"\nLoaded DataFrame for RecordSet @id: {record_set_id}")
        print(f"Columns (Field/Column @id): {df.columns.tolist()}")
        display(df.head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific numeric criteria, normalizing fields, and grouping, always referencing fields by their `@id`.

**Choose an example RecordSet and a numeric field for demo:**

In [ ]:
# Example: Use the first RecordSet and a numeric column if it exists
if len(dataframes) == 0:
    print("No dataframes loaded: cannot proceed with EDA.")
else:
    example_rs_id = list(dataframes.keys())[0]
    df = dataframes[example_rs_id]
    # Try to find a likely numeric column (heuristic: column name mentions 'abundance', 'MW', 'weight', etc.)
    import numpy as np
    numeric_cols = [col for col in df.columns if df[col].dtype in [np.float64, np.float32, np.int64, np.int32]]
    # Otherwise, try another heuristic based on column names
    if not numeric_cols:
        for kw in ['abundance', 'MW', 'weight', 'count', 'coverage', 'peptide', 'intensity']:
            matches = [col for col in df.columns if kw in col.lower()]
            if matches:
                numeric_cols.extend(matches)
                break
    if numeric_cols:
        numeric_field_id = numeric_cols[0]
        print(f"Using numeric field (@id): {numeric_field_id}")
        print(df[[numeric_field_id]].head())

        # Convert column to numeric, coercing errors
        df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
        threshold = df[numeric_field_id].mean()  # use mean as threshold for demonstration
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records where {numeric_field_id} > {threshold:.2f}:")
        display(filtered_df[[numeric_field_id]].head())

        # Normalization
        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
            filtered_df[numeric_field_id].std()
        )
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try to find a group-by field (heuristic: 'accession', 'protein', 'sample', ...)
        group_field_candidates = [c for c in df.columns if any(t in c.lower() for t in ['sample', 'accession', 'group', 'type', 'category'])]
        if group_field_candidates:
            group_field = group_field_candidates[0]
            print(f"Grouping by field (@id): {group_field}")
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            display(grouped_df.head())
        else:
            print("No suitable group field found.")
    else:
        print("No numeric field found for EDA.")

## 5. Visualization
Visualize the distribution of the selected numeric field and/or relationships, always referencing via the `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if len(dataframes) > 0 and 'numeric_field_id' in locals():
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), bins=40, kde=True)
    plt.xlabel(numeric_field_id)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.show()

    # If a group field is detected earlier
    if 'group_field' in locals():
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=df[group_field], y=df[numeric_field_id])
        plt.title(f"{numeric_field_id} across {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, we've demonstrated how to load and explore a FAIR-compliant Croissant dataset, referencing all structural elements by `@id`, using the `mlcroissant` library. We've performed filtering, normalization, and basic visualization on a numeric field. For deeper and domain-specific analyses, refer to the full field and record set schema and adapt the code to your research question.